In [ ]:
import csv
import json
import random
from collections import defaultdict
from typing import DefaultDict

# =========================
# CONFIG
# =========================
FLICKR30K_DIR = ".."
DATA_DIR = f"{FLICKR30K_DIR}/data"

INPUT_FILE = f"{FLICKR30K_DIR}/flickr30k-dataset/captions.txt"
OUTPUT_FILE = f"{DATA_DIR}/captions_split.json"

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

random.seed(42)


# =========================
# FUNCTIONS
# =========================
def load_captions(input_file: str) -> dict[str, list[str]]:
    """Load captions and group them by image."""

    image_to_captions: DefaultDict[str, list[str]] = defaultdict(list)

    with open(input_file, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # Skip header

        for row in reader:
            if len(row) < 2:
                continue

            image = row[0].strip().removesuffix(".jpg")
            caption = row[1].strip()

            image_to_captions[image].append(caption)

    return dict(image_to_captions)


def split_images(images: list[str]) -> dict[str, list[str]]:
    """Random train/val/test split."""

    images = images.copy()
    random.shuffle(images)

    n = len(images)

    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    return {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train + n_val:]
    }


def build_dataset(
    image_to_captions: dict[str, list[str]],
    splits: dict[str, list[str]]
) -> dict[str, dict[str, list[str]]]:
    """Build dataset for JSON export."""

    return {
        split: {
            image: image_to_captions[image]
            for image in image_list
        }
        for split, image_list in splits.items()
    }


def save_json(
    dataset: dict[str, dict[str, list[str]]],
    output_file: str
) -> None:
    """Save dataset as JSON."""

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)


def save_image_lists(
    splits: dict[str, list[str]],
    output_dir: str
) -> None:
    """Save train/val/test image lists."""

    for split, image_list in splits.items():
        with open(
            f"{output_dir}/{split}_images.txt",
            "w",
            encoding="utf-8"
        ) as f:
            f.write("\n".join(image_list))


# =========================
# MAIN
# =========================
image_to_captions = load_captions(INPUT_FILE)

splits = split_images(list(image_to_captions.keys()))

dataset = build_dataset(image_to_captions, splits)

save_json(dataset, OUTPUT_FILE)
save_image_lists(splits, DATA_DIR)

print("Dataset created successfully.")
print(f"Train: {len(splits['train'])}")
print(f"Validation: {len(splits['val'])}")
print(f"Test: {len(splits['test'])}")